# Validador de Procesos contra Arquetipos y Mapa de Capacidades

Este notebook carga un archivo de actividades de procesos, lo cruza contra el catálogo de arquetipos para identificar a cuál arquetipo pertenece cada código de proceso, y luego busca coincidencias entre las actividades y el mapa de capacidades de procesos.

## 1. Importación de librerías

Se importa `pandas`, la librería principal para manipulación de datos tabulares en este notebook.

In [28]:
import pandas as pd


## 2. Carga del archivo de actividades

Lee el archivo `data/query_actividades_proc_q_actual.csv` y lo carga en el DataFrame `df`.

In [29]:
from pathlib import Path
import pandas as pd

csv_path = Path("data/query_actividades_proc_q_actual.csv")
df = pd.read_csv(csv_path, dtype=str)

print(f"Archivo cargado: {csv_path}")
print(f"{len(df)} filas · {len(df.columns)} columnas")
print(f"Columnas: {list(df.columns)}")
df.head()

Archivo cargado: data\query_actividades_proc_q_actual.csv
39033 filas · 5 columnas
Columnas: ['t1.id_epica', 't1.periodo', 'codigo_proceso', 't2.procedimiento', 't2.actividad']


,t1.id_epica,t1.periodo,codigo_proceso,t2.procedimiento,t2.actividad
0,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,bloqueo usuario
1,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,solicitar impugnación
2,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,solicitar autorización
3,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,s: crear o vincular clientes
4,6788253,2026Q1,FID050210,Definir y ejecutar estrategia de Asset Management,revisión inconsistencias


## 3. Códigos de proceso únicos × Arquetipos (tabla expandida)

Extrae los `codigo_proceso` únicos del CSV de actividades y los cruza contra `arquetipos.csv`.  
El resultado tiene **una fila por cada par (codigo_proceso, arquetipo)**: un proceso puede aparecer en varios arquetipos y un arquetipo puede contener varios procesos.

In [30]:
import sys, os, importlib
sys.path.insert(0, os.getcwd())

import helper.match_arquetipos as _ma
importlib.reload(_ma)
from helper.match_arquetipos import load_arquetipos

# ── Guard: carga df si la celda 2 no fue ejecutada ───────────────────────────
if 'df' not in vars() or 'codigo_proceso' not in df.columns:
    from pathlib import Path
    import pandas as pd
    df = pd.read_csv(Path('data/query_actividades_proc_q_actual.csv'), dtype=str)
    print("(df recargado desde archivos fuente)")

# ── 1. Códigos de proceso únicos ─────────────────────────────────────────────
df_codigos = (
    df[['codigo_proceso']]
    .drop_duplicates()
    .assign(codigo_proceso=lambda x: x['codigo_proceso'].astype(str).str.strip())
    .reset_index(drop=True)
)

# ── 2. Arquetipos explodidos (una fila por código de proceso en el catálogo) ──
_, df_arq_expl = load_arquetipos('data/arquetipos.csv')

arq_code_col = next(c for c in df_arq_expl.columns if 'Código' in c and 'Arquetipo' in c)
arq_name_col = next(c for c in df_arq_expl.columns if 'Nombre' in c and 'Arquetipo' in c)

# ── 3. Left join: conserva todos los códigos, N/A si no hay arquetipo ────────
df_proc_arquetipos = (
    df_codigos
    .merge(
        df_arq_expl[['proc_code', arq_code_col, arq_name_col]].drop_duplicates(),
        left_on='codigo_proceso',
        right_on='proc_code',
        how='left',
    )
    [['codigo_proceso', arq_code_col, arq_name_col]]
    .fillna('N/A')
    .sort_values(['codigo_proceso', arq_code_col])
    .reset_index(drop=True)
)

sin_arquetipo = (df_proc_arquetipos[arq_code_col] == 'N/A').sum()
print(f"Códigos únicos en el CSV:            {df['codigo_proceso'].nunique()}")
print(f"Con al menos un arquetipo asociado:  {df['codigo_proceso'].nunique() - sin_arquetipo}")
print(f"Sin ningún arquetipo (N/A):          {sin_arquetipo}")
print(f"Filas en la tabla expandida:         {len(df_proc_arquetipos)}")
df_proc_arquetipos

Códigos únicos en el CSV:            168
Con al menos un arquetipo asociado:  80
Sin ningún arquetipo (N/A):          88
Filas en la tabla expandida:         187


,codigo_proceso,Código Arquetipo,Nombre Arquetipo
0,FID050210,N/A,N/A
1,FID080105,N/A,N/A
2,PAN040102,ARQ002,Abrir producto
3,PAN040103,ARQ002,Abrir producto
4,PAN040107,ARQ002,Abrir producto
...,...,...,...
182,T160668,N/A,N/A
183,T160669,ARQ002,Abrir producto
184,T160673,ARQ009,Autorizar tokenización
185,T160675,ARQ002,Abrir producto


In [31]:
import os

output_path = 'data/output/proc_arquetipos.csv'
os.makedirs('data/output', exist_ok=True)
df_proc_arquetipos.to_csv(output_path, index=False, encoding='utf-8')
print(f"Guardado en: {output_path}")
print(f"Total de filas: {len(df_proc_arquetipos)}")

Guardado en: data/output/proc_arquetipos.csv
Total de filas: 187


## 4. Arquetipos × Capacidades de Proceso (tabla expandida)

Cruza los `Nombre Arquetipo` únicos del paso anterior contra el archivo `mapa_capacidades.csv`, evaluando si el nombre del arquetipo aparece **dentro** del campo `Arquetipos Relacionado` (búsqueda por substring, ya que el campo puede contener uno o varios arquetipos).

De cada fila coincidente se extraen:
- `Capacidad de Procesos (Nivel 3)` → renombrado como `capacidades`
- `Dominio`
- `Subdominio Nivel 1`
- `Subdominio Nivel 2`

El resultado `df_arq_capacidades` tiene **una fila por cada par (Nombre Arquetipo, capacidad)**.

In [32]:
import sys, os, importlib, pandas as pd
sys.path.insert(0, os.getcwd())

# ── Guard: reconstruye df_proc_arquetipos si la celda 3 no fue ejecutada ─────
if 'df_proc_arquetipos' not in vars() or 'arq_name_col' not in vars():
    import helper.match_arquetipos as _ma
    importlib.reload(_ma)
    from helper.match_arquetipos import load_arquetipos
    from pathlib import Path

    _df = pd.read_csv(Path('data/query_actividades_proc_q_actual.csv'), dtype=str)
    _, _df_arq_expl = load_arquetipos('data/arquetipos.csv')

    arq_code_col = next(c for c in _df_arq_expl.columns if 'Código' in c and 'Arquetipo' in c)
    arq_name_col = next(c for c in _df_arq_expl.columns if 'Nombre' in c and 'Arquetipo' in c)

    _df_codigos = (
        _df[['codigo_proceso']]
        .drop_duplicates()
        .assign(codigo_proceso=lambda x: x['codigo_proceso'].astype(str).str.strip())
        .reset_index(drop=True)
    )
    df_proc_arquetipos = (
        _df_codigos
        .merge(
            _df_arq_expl[['proc_code', arq_code_col, arq_name_col]].drop_duplicates(),
            left_on='codigo_proceso', right_on='proc_code', how='left',
        )
        [['codigo_proceso', arq_code_col, arq_name_col]]
        .fillna('N/A')
        .reset_index(drop=True)
    )
    print("(df_proc_arquetipos reconstruido desde archivos fuente)")

# ── Carga del mapa de capacidades (UTF-8 con BOM) ────────────────────────────
df_cap = pd.read_csv('data/mapa_capacidades.csv', dtype=str, encoding='utf-8-sig')

_cap_cols = [
    'Capacidad de Procesos (Nivel 3)',
    'Dominio',
    'Subdominio Nivel 1',
    'Subdominio Nivel 2',
    'Arquetipos Relacionado',
]
df_cap_clean = (
    df_cap[_cap_cols]
    .dropna(subset=['Arquetipos Relacionado', 'Capacidad de Procesos (Nivel 3)'])
    .assign(**{'Arquetipos Relacionado': lambda x: x['Arquetipos Relacionado'].str.strip()})
    .reset_index(drop=True)
)

# ── Nombres de arquetipo únicos a buscar (excluye N/A) ───────────────────────
nombres_arq = [n for n in df_proc_arquetipos[arq_name_col].unique() if n != 'N/A']

# ── Cruce por substring: una fila por (arquetipo, capacidad) ─────────────────
fragmentos = []
for nombre in nombres_arq:
    mask = df_cap_clean['Arquetipos Relacionado'].str.contains(nombre, regex=False, na=False)
    coincidencias = df_cap_clean[mask].copy()
    coincidencias['Nombre Arquetipo'] = nombre
    fragmentos.append(coincidencias)

df_arq_capacidades = (
    pd.concat(fragmentos, ignore_index=True)
    .rename(columns={'Capacidad de Procesos (Nivel 3)': 'capacidades'})
    [['Nombre Arquetipo', 'capacidades', 'Dominio', 'Subdominio Nivel 1', 'Subdominio Nivel 2']]
    .drop_duplicates()
    .sort_values(['Nombre Arquetipo', 'capacidades'])
    .reset_index(drop=True)
)

print(f"Arquetipos únicos buscados:          {len(nombres_arq)}")
print(f"Con al menos una capacidad asociada: {df_arq_capacidades['Nombre Arquetipo'].nunique()}")
print(f"Filas en la tabla expandida:         {len(df_arq_capacidades)}")
df_arq_capacidades

Arquetipos únicos buscados:          17
Con al menos una capacidad asociada: 8
Filas en la tabla expandida:         259


,Nombre Arquetipo,capacidades,Dominio,Subdominio Nivel 1,Subdominio Nivel 2
0,Abrir producto,Ajustar parámetro,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...","Tarjeta crédito, tarjeta débito, cheques, reca..."
1,Abrir producto,Asignar sucursal de radicación,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...",NaN
2,Abrir producto,Asociar cuenta al producto,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...","Depósitos a la vista, depósitos a plazo, tarje..."
3,Abrir producto,Asociar producto al cliente,Clientes,Relacionamiento con clientes,Por definir
4,Abrir producto,Asociar producto al cliente,"Negocios, habilitadores de los negocios","Captaciones, medios de pago, soluciones para e...",Por definir
...,...,...,...,...,...
254,Realizar Vinculación,Registrar información en fuentes oficiales,Prueba dominio dueño,Prueba sudominio N1,Prueba subdominio N2
255,Realizar Vinculación,Segmentar cliente,Prueba dominio dueño,Prueba sudominio N1,Prueba subdominio N2
256,Realizar Vinculación,Validar procesos de vinculación en curso,Prueba dominio dueño,Prueba sudominio N1,Prueba subdominio N2
257,Realizar Vinculación,Verificar integridad y validez de información ...,Prueba dominio dueño,Prueba sudominio N1,Prueba subdominio N2


In [33]:
import os

output_path = 'data/output/arq_capacidades.csv'
os.makedirs('data/output', exist_ok=True)
df_arq_capacidades.to_csv(output_path, index=False, encoding='utf-8')
print(f"Guardado en: {output_path}")
print(f"Total de filas: {len(df_arq_capacidades)}")

Guardado en: data/output/arq_capacidades.csv
Total de filas: 259


## 4b. Actividades con correspondencia en Capacidades (trazabilidad)

Antes de agregar métricas, se identifica exactamente qué actividades de cada procedimiento tienen match con algún valor del campo `capacidades` generado en el paso 4.

- La búsqueda es normalizada (minúsculas, sin punto final) y evalúa tres modalidades: exacta, actividad ⊂ capacidad, capacidad ⊂ actividad.
- Se conservan únicamente las filas con match encontrado, con una fila por actividad única cubierta.
- El resultado `df_actividades_con_match` es persistido en `data/output/` para trazabilidad y auditoria, y es reutilizado directamente por el paso 5.

In [34]:
import pandas as pd, sys, os, importlib

# ── Guard: reconstruye df_arq_capacidades si la celda 4 no fue ejecutada ─────
if 'df_arq_capacidades' not in vars():
    sys.path.insert(0, os.getcwd())
    import helper.match_arquetipos as _ma; importlib.reload(_ma)
    from helper.match_arquetipos import load_arquetipos
    from pathlib import Path

    _df_raw = pd.read_csv(Path('data/query_actividades_proc_q_actual.csv'), dtype=str)
    _, _arq_expl = load_arquetipos('data/arquetipos.csv')
    _name_col = next(c for c in _arq_expl.columns if 'Nombre' in c and 'Arquetipo' in c)
    _code_col = next(c for c in _arq_expl.columns if 'Código' in c and 'Arquetipo' in c)
    _df_proc = (
        _df_raw[['codigo_proceso']].drop_duplicates()
        .assign(codigo_proceso=lambda x: x['codigo_proceso'].astype(str).str.strip())
        .merge(_arq_expl[['proc_code', _code_col, _name_col]].drop_duplicates(),
               left_on='codigo_proceso', right_on='proc_code', how='left')
        [[_name_col]].fillna('N/A')
    )
    _names = [n for n in _df_proc[_name_col].unique() if n != 'N/A']
    _cap = pd.read_csv('data/mapa_capacidades.csv', dtype=str, encoding='utf-8-sig')
    _cap_clean = (
        _cap[['Capacidad de Procesos (Nivel 3)', 'Arquetipos Relacionado']]
        .dropna(subset=['Arquetipos Relacionado', 'Capacidad de Procesos (Nivel 3)'])
        .assign(**{'Arquetipos Relacionado': lambda x: x['Arquetipos Relacionado'].str.strip()})
    )
    _frags = []
    for _n in _names:
        _m = _cap_clean[_cap_clean['Arquetipos Relacionado'].str.contains(_n, regex=False, na=False)].copy()
        _m['Nombre Arquetipo'] = _n
        _frags.append(_m)
    df_arq_capacidades = (
        pd.concat(_frags, ignore_index=True)
        .rename(columns={'Capacidad de Procesos (Nivel 3)': 'capacidades'})
        [['capacidades']]
        .drop_duplicates().reset_index(drop=True)
    )
    print("(df_arq_capacidades reconstruido desde archivos fuente)")

# ── 1. Normalización ──────────────────────────────────────────────────────────
def _norm(texto):
    if pd.isna(texto) or not str(texto).strip():
        return ''
    return str(texto).strip().lower().rstrip('.')

caps_norm = {_norm(c) for c in df_arq_capacidades['capacidades'].dropna().unique() if _norm(c)}

# ── 2. Actividades únicas por (proceso, procedimiento) ───────────────────────
df_act = pd.read_csv('data/query_actividades_proc_q_actual.csv', dtype=str)

df_unique_act = (
    df_act[['codigo_proceso', 't2.procedimiento', 't2.actividad']]
    .drop_duplicates()
    .reset_index(drop=True)
)

# ── 3. Filtrar solo las actividades con match ─────────────────────────────────
def _tiene_match(actividad):
    act = _norm(actividad)
    if not act:
        return False
    return any(act == cap or act in cap or cap in act for cap in caps_norm)

df_actividades_con_match = (
    df_unique_act[df_unique_act['t2.actividad'].apply(_tiene_match)]
    .rename(columns={'t2.actividad': 'match de actividad encontrada'})
    .reset_index(drop=True)
)

# ── 4. Persistir ──────────────────────────────────────────────────────────────
os.makedirs('data/output', exist_ok=True)
output_path = 'data/output/actividades_con_match.csv'
df_actividades_con_match.to_csv(output_path, index=False, encoding='utf-8')

print(f"Actividades únicas con match:        {len(df_actividades_con_match)}")
print(f"Procesos con al menos un match:      {df_actividades_con_match['codigo_proceso'].nunique()}")
print(f"Guardado en: {output_path}")
df_actividades_con_match

Actividades únicas con match:        167
Procesos con al menos un match:      57
Guardado en: data/output/actividades_con_match.csv


,codigo_proceso,t2.procedimiento,match de actividad encontrada
0,FID050210,Definir y ejecutar estrategia de Asset Management,solicitar autorización
1,T050402,T050402 Gestionar requerimientos legal desemba...,aplicar reglas de negocio
2,T050402,Gestionar requerimientos legales de embargos y...,cmp 546 generar documento
3,T050402,Gestionar requerimientos legal desembargo sobr...,aplicar reglas de negocio
4,T050402,Gestionar solicitudes,cmp 546 generar documento
...,...,...,...
162,VAL080512,Procedimiento asesorar clientes MK Valores y F...,solicitar autorización
163,VAL080512,Procedimiento asesorar clientes MK Valores y F...,consultar información [act003-03]
164,VAL080512,Procedimiento monitorear recomendaciones profe...,obtener información
165,VAL080512,Procedimiento asesorar clientes MK Valores y F...,consultar información [act003-02]


## 5. Cobertura de actividades vs. Capacidades de Proceso

Agrupa el resultado del paso 4b para calcular métricas de cobertura por `codigo_proceso` + `t2.procedimiento`.

- **total actividades**: actividades distintas del procedimiento (sobre el CSV original).
- **# match encontradas**: actividades distintas con cobertura (desde `df_actividades_con_match`).
- **% match actividad/capacidades**: `# match / total actividades`.

In [35]:
import pandas as pd, os

# ── Guard: carga df_actividades_con_match desde CSV si el paso 4b no corrió ──
if 'df_actividades_con_match' not in vars():
    df_actividades_con_match = pd.read_csv(
        'data/output/actividades_con_match.csv', dtype=str
    )
    print("(df_actividades_con_match cargado desde data/output/actividades_con_match.csv)")

# ── 1. Total de actividades distintas por (proceso, procedimiento) ────────────
df_act = pd.read_csv('data/query_actividades_proc_q_actual.csv', dtype=str)

total_act = (
    df_act[['codigo_proceso', 't2.procedimiento', 't2.actividad']]
    .drop_duplicates()
    .groupby(['codigo_proceso', 't2.procedimiento'])['t2.actividad']
    .nunique()
    .rename('total actividades')
)

# ── 2. Matches distintos por (proceso, procedimiento) — desde paso 4b ─────────
match_act = (
    df_actividades_con_match
    .groupby(['codigo_proceso', 't2.procedimiento'])['match de actividad encontrada']
    .nunique()
    .rename('# match encontradas')
)

# ── 3. Unir y calcular porcentaje ─────────────────────────────────────────────
cobertura = (
    total_act.to_frame()
    .join(match_act, how='left')
    .fillna({'# match encontradas': 0})
    .assign(**{'# match encontradas': lambda x: x['# match encontradas'].astype(int)})
    .assign(**{'% match actividad/capacidades': lambda x:
        (x['# match encontradas'] / x['total actividades'] * 100).round(1).astype(str) + '%'})
    .reset_index()
    .sort_values(['codigo_proceso', 't2.procedimiento'])
    .reset_index(drop=True)
)

print(f"Combinaciones (proceso × procedimiento): {len(cobertura)}")
print(f"Con al menos un match:                   {(cobertura['# match encontradas'] > 0).sum()}")
print(f"Sin ningún match:                        {(cobertura['# match encontradas'] == 0).sum()}")
cobertura

Combinaciones (proceso × procedimiento): 808
Con al menos un match:                   121
Sin ningún match:                        687


,codigo_proceso,t2.procedimiento,total actividades,# match encontradas,% match actividad/capacidades
0,FID050210,Definir y ejecutar estrategia de Asset Management,61,1,1.6%
1,FID080105,Controlar y Monitorear Políticas y Límites Rie...,12,0,0.0%
2,PAN040102,Abrir Producto de Crédito,4,0,0.0%
3,PAN040102,Autorizar Operaciones por Callback,6,0,0.0%
4,PAN040102,Crear Cuenta y Autorizar desembolsos,55,0,0.0%
...,...,...,...,...,...
803,T160675,Personalizar y entregar Código QR,19,0,0.0%
804,VAL080512,"Asesorar, negociar o promover operaciones con ...",70,0,0.0%
805,VAL080512,Habilitar comerciales mesa de dinero,7,0,0.0%
806,VAL080512,Procedimiento asesorar clientes MK Valores y F...,30,4,13.3%


In [36]:
import os

output_path = 'data/output/cobertura_actividades_capacidades.csv'
os.makedirs('data/output', exist_ok=True)
cobertura.to_csv(output_path, index=False, encoding='utf-8')
print(f"Guardado en: {output_path}")
print(f"Total de filas: {len(cobertura)}")

Guardado en: data/output/cobertura_actividades_capacidades.csv
Total de filas: 808
